# Demo 04 : Node failures

Run the cells in order and watch. There is nothing to write.

## a · retry-policy

We inject a transient inventory failure. `max_attempts` includes the initial attempt, and the retry filter selects the exceptions that may repeat.

In [1]:
from typing_extensions import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.types import RetryPolicy


class LookupState(TypedDict):
    part_no: str
    status: str


def inventory(state: LookupState, runtime: Runtime):
    attempt = runtime.execution_info.node_attempt
    print("attempt:", attempt)
    if attempt < 3:
        raise ConnectionError("inventory temporarily unavailable")
    return {"status": "inventory available"}


builder = StateGraph(LookupState)
builder.add_node("inventory", inventory, retry_policy=RetryPolicy(
    max_attempts=3, initial_interval=0.01, jitter=False, retry_on=ConnectionError))
builder.add_edge(START, "inventory")
builder.add_edge("inventory", END)
print(builder.compile().invoke({"part_no": "CE-DA-03", "status": "pending"}))

attempt: 1
attempt: 2
attempt: 3
{'part_no': 'CE-DA-03', 'status': 'inventory available'}


## b · node-timeout

We run async nodes with run and idle limits. Heartbeats refresh the idle clock, while the run limit remains fixed. Expected timeout exceptions are caught and printed.

In [2]:
import asyncio
from langgraph.errors import NodeTimeoutError
from langgraph.types import TimeoutPolicy


async def slow_inventory(state: LookupState, runtime: Runtime):
    try:
        await asyncio.sleep(0.5)
        return {"status": "late"}
    finally:
        print("slow node: cancelled or finished")


async def progressing_inventory(state: LookupState, runtime: Runtime):
    for batch in range(4):
        await asyncio.sleep(0.03)
        runtime.heartbeat()
    return {"status": "completed with heartbeats"}


async def timed(node, policy):
    builder = StateGraph(LookupState)
    builder.add_node("inventory", node, timeout=policy)
    builder.add_edge(START, "inventory")
    builder.add_edge("inventory", END)
    try:
        print(await builder.compile().ainvoke({"part_no": "CE-DA-03", "status": "pending"}))
    except NodeTimeoutError as exc:
        print("timeout:", exc.kind, "node:", exc.node)


await timed(slow_inventory, TimeoutPolicy(run_timeout=0.08))
await timed(slow_inventory, TimeoutPolicy(idle_timeout=0.08, refresh_on="heartbeat"))
await timed(progressing_inventory, TimeoutPolicy(idle_timeout=0.08, refresh_on="heartbeat"))
await timed(progressing_inventory, TimeoutPolicy(run_timeout=0.08, idle_timeout=0.08,
                                                refresh_on="heartbeat"))

try:
    invalid = StateGraph(LookupState)
    invalid.add_node("inventory", inventory, timeout=0.08)
    invalid.add_edge(START, "inventory")
    invalid.compile()
except ValueError as exc:
    print("sync timeout:", str(exc).splitlines()[0])

slow node: cancelled or finished
timeout: run node: inventory
slow node: cancelled or finished
timeout: idle node: inventory


{'part_no': 'CE-DA-03', 'status': 'completed with heartbeats'}
timeout: run node: inventory
sync timeout: Node timeouts are only supported for async nodes because sync Python execution cannot be safely cancelled in-process. Node 'inventory' is sync.


## c · error-handler

We route an exhausted failure to a manual queue. The handler receives state and an injected `NodeError`. A failed attempt's state writes are discarded, but external side effects need their own protection.

In [3]:
from langgraph.errors import NodeError
from langgraph.types import Command


def unavailable(state: LookupState, runtime: Runtime):
    print("attempt:", runtime.execution_info.node_attempt)
    raise ConnectionError("inventory unavailable")


def on_error(state: LookupState, error: NodeError) -> Command:
    print("handler:", error.node, type(error.error).__name__)
    return Command(update={"status": "manual stock check required"}, goto="notify")


def notify(state: LookupState):
    print("queue:", state["part_no"], state["status"])
    return {}


builder = StateGraph(LookupState)
builder.add_node("inventory", unavailable, error_handler=on_error,
                 retry_policy=RetryPolicy(max_attempts=2, retry_on=ConnectionError,
                                          initial_interval=0.01, jitter=False))
builder.add_node("notify", notify)
builder.add_edge(START, "inventory")
builder.add_edge("inventory", END)
builder.add_edge("notify", END)
print(builder.compile().invoke({"part_no": "CE-DA-03", "status": "pending"}))

attempt: 1
attempt: 2
handler: inventory ConnectionError
queue: CE-DA-03 manual stock check required
{'part_no': 'CE-DA-03', 'status': 'manual stock check required'}


We combine an async timeout with the default retry filter and the same recovery handler. Each retry gets a fresh timeout budget, and `NodeTimeoutError` reaches the handler after retries are exhausted.

In [4]:
async def timed_inventory(state: LookupState, runtime: Runtime):
    print("timed attempt:", runtime.execution_info.node_attempt)
    await asyncio.sleep(0.5)
    return {"status": "late"}


builder = StateGraph(LookupState)
builder.add_node("inventory", timed_inventory, timeout=TimeoutPolicy(run_timeout=0.08),
                 retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.01, jitter=False),
                 error_handler=on_error)
builder.add_node("notify", notify)
builder.add_edge(START, "inventory")
builder.add_edge("inventory", END)
builder.add_edge("notify", END)
print(await builder.compile().ainvoke({"part_no": "CE-DA-03", "status": "pending"}))

timed attempt: 1
timed attempt: 2


handler: inventory NodeTimeoutError
queue: CE-DA-03 manual stock check required
{'part_no': 'CE-DA-03', 'status': 'manual stock check required'}


## Things to try

Optional : change the bus or part identifier and run the example again.